# Day 02 — Data Ingestion
**Member 1** | Goal: Fetch weather data for all 15 districts and load cotton dataset into DuckDB

## Objectives
- Run the ingestion pipeline
- Verify weather CSV files created for all 15 districts
- Verify DuckDB tables `raw_weather` and `raw_cotton` created
- Inspect raw data shape and sample rows

In [1]:
import pandas as pd
import duckdb
import os, sys
sys.path.append('../src')
from config import DB_PATH, RAW_WEATHER_DIR, LOCATIONS

print(f'DB path:          {DB_PATH}')
print(f'Weather CSV dir:  {RAW_WEATHER_DIR}')

DB path:          C:\Users\User\Desktop\m5-project-weather-pipeline\data\cotton_project.duckdb
Weather CSV dir:  C:\Users\User\Desktop\m5-project-weather-pipeline\data\raw\weather


## 1. Run Ingestion Pipeline

In [2]:
from ingestion import run_ingestion

# This fetches weather from Open-Meteo API for all 15 districts
# and loads cotton Excel into DuckDB.
# Incremental: skips already-fetched stations.
run_ingestion()

[2026-04-29 09:39:03] 
[2026-04-29 09:39:03] MEMBER 1 — DATA INGESTION PIPELINE
[2026-04-29 09:39:03] =======================================================
[2026-04-29 09:39:03] =======================================================
[2026-04-29 09:39:03] STEP 1 — Weather Ingestion (Incremental)
[2026-04-29 09:39:03] =======================================================
[2026-04-29 09:39:03]   Goranboy district: already up to date (last year=2026) ✓ skipping
[2026-04-29 09:39:03]   Kurdamir district: already up to date (last year=2026) ✓ skipping
[2026-04-29 09:39:03]   Yevlakh district: already up to date (last year=2026) ✓ skipping
[2026-04-29 09:39:03]   Zardab district: already up to date (last year=2026) ✓ skipping
[2026-04-29 09:39:04]   Tartar district: already up to date (last year=2026) ✓ skipping
[2026-04-29 09:39:04]   Aghdam district: already up to date (last year=2026) ✓ skipping
[2026-04-29 09:39:04]   Sabirabad district: already up to date (last year=2026) ✓ skipping

## 2. Verify Weather CSV Files

In [3]:
csv_files = [f for f in os.listdir(RAW_WEATHER_DIR) if f.endswith('.csv')]
print(f'CSV files found: {len(csv_files)} / {len(LOCATIONS)} expected')
print()
for f in sorted(csv_files):
    path = os.path.join(RAW_WEATHER_DIR, f)
    df   = pd.read_csv(path)
    print(f'  {f:<45} {len(df):>6} rows  years {df.year.min()}–{df.year.max()}')

CSV files found: 15 / 15 expected

  aghdam_district_daily.csv                       9615 rows  years 1999–2026
  aghjabadi_district_daily.csv                    9615 rows  years 1999–2026
  barda_district_daily.csv                        9615 rows  years 1999–2026
  beylagan_district_daily.csv                     9615 rows  years 1999–2026
  bilasuvar_district_daily.csv                    9615 rows  years 1999–2026
  goranboy_district_daily.csv                     9615 rows  years 1999–2026
  imishli_district_daily.csv                      9615 rows  years 1999–2026
  kurdamir_district_daily.csv                     9615 rows  years 1999–2026
  neftchala_district_daily.csv                    9615 rows  years 1999–2026
  saatli_district_daily.csv                       9615 rows  years 1999–2026
  sabirabad_district_daily.csv                    9615 rows  years 1999–2026
  salyan_district_daily.csv                       9615 rows  years 1999–2026
  tartar_district_daily.csv              

## 3. Inspect DuckDB Tables

In [4]:
con    = duckdb.connect(DB_PATH)
tables = con.execute('SHOW TABLES').fetchall()
print('Tables in DuckDB:')
for (t,) in tables:
    count = con.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
    cols  = len(con.execute(f'SELECT * FROM {t} LIMIT 1').description)
    print(f'  {t:<25} {count:>6} rows  {cols:>3} cols')

Tables in DuckDB:
  clean_cotton                 375 rows    4 cols
  clean_weather             142455 rows    9 cols
  features                     375 rows   44 cols
  features_with_risk           375 rows   55 cols
  predictions                   15 rows    9 cols
  raw_cotton                   725 rows    3 cols
  raw_weather               144225 rows    9 cols


## 4. Sample raw_weather

In [5]:
df_w = con.execute("SELECT * FROM raw_weather LIMIT 5").df()
df_w

,date,temp_mean,precipitation,humidity_mean,wind_speed,region,year,month,day
0,1999-12-31 20:00:00,7.481499,0.0,73.393830,7.594207,Goranboy district,1999,12,31
1,2000-01-01 20:00:00,8.079417,0.0,63.938328,7.289444,Goranboy district,2000,1,1
2,2000-01-02 20:00:00,5.275250,0.0,88.345770,7.091177,Goranboy district,2000,1,2
3,2000-01-03 20:00:00,5.156499,1.4,88.488850,8.707237,Goranboy district,2000,1,3
4,2000-01-04 20:00:00,5.256500,7.8,94.401460,12.599998,Goranboy district,2000,1,4


## 5. Sample raw_cotton

In [6]:
df_c = con.execute("SELECT * FROM raw_cotton ORDER BY region, year LIMIT 10").df()
df_c

,region,year,yield_tonnes
0,Agdash district,2000,6.9
1,Agdash district,2001,9.3
2,Agdash district,2002,19.8
3,Agdash district,2003,14.5
4,Agdash district,2004,14.4
5,Agdash district,2005,15.1
6,Agdash district,2006,15.5
7,Agdash district,2007,15.5
8,Agdash district,2008,6.9
9,Agdash district,2009,11.8


## 6. Weather Coverage Check

In [7]:
coverage = con.execute("""
    SELECT region,
           MIN(year) AS first_year,
           MAX(year) AS last_year,
           COUNT(DISTINCT year) AS years_covered,
           COUNT(*) AS total_rows
    FROM raw_weather
    GROUP BY region
    ORDER BY region
""").df()
coverage

,region,first_year,last_year,years_covered,total_rows
0,Aghdam district,1999,2026,28,9615
1,Aghjabadi district,1999,2026,28,9615
2,Barda district,1999,2026,28,9615
3,Beylagan district,1999,2026,28,9615
4,Bilasuvar district,1999,2026,28,9615
5,Goranboy district,1999,2026,28,9615
6,Imishli district,1999,2026,28,9615
7,Kurdamir district,1999,2026,28,9615
8,Neftchala district,1999,2026,28,9615
9,Saatli district,1999,2026,28,9615


## 7. Null Check on raw_weather

In [8]:
df_all = con.execute('SELECT * FROM raw_weather').df()
null_summary = df_all.isnull().sum()
print('Null counts per column:')
print(null_summary[null_summary > 0] if null_summary.sum() > 0 else 'No nulls found ✓')
con.close()

Null counts per column:
No nulls found ✓


## Summary
- Weather fetched for all 15 districts (2000–2025) ✓
- `raw_weather` table created in DuckDB ✓
- `raw_cotton` table created in DuckDB ✓
- **Next:** `day_03_database.ipynb` — database layer verification